# IT2011 - Artificial Intelligence and Machine Learning
## Master Integrated Data Preprocessing & Pipeline Deliverable
### Group ID: `2026-Y2-S1-MET-23`
### Academic Year: Year 2, Semester 1 (2026) — SLIIT Faculty of Computing

---

### Pipeline Architecture & Group Integration (5 Marks Shared)

This notebook represents the unified, production-ready preprocessing pipeline integrating the distinct techniques of all 6 team members:

```
[Raw CSV: 46,173 Reviews]
   │
   ├─► Stage 1 (Member 1 - IT25102549): Text Cleaning, HTML/URL Stripping & Contraction Expansion
   │
   ├─► Stage 2 (Member 2 - IT25102550): Domain Stopword Filtering & Lemmatization
   │
   ├─► Stage 3 (Member 3 - IT25102631): Multi-Label Categorical Binarization on Movie Genres
   │
   ├─► Stage 4 (Member 4 - IT25102877): Review Length IQR Winsorization & Numerical Scaling
   │
   ├─► Stage 5 (Member 5 - IT25103066): Multiclass Imbalance Quantification & Balanced Loss Weights
   │
   └─► Stage 6 (Member 6 - IT25103132): TF-IDF Vectorization & TruncatedSVD Feature Extraction
   │
   ▼
[Export: results/outputs/processed_movie_reviews.csv] ──► Ready for Phase 2 Model Training
```


In [1]:
import os
import re
import ast
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer, RobustScaler, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Ensure directory structure exists
os.makedirs('results/outputs', exist_ok=True)
os.makedirs('results/logs', exist_ok=True)
os.makedirs('results/eda_visualizations', exist_ok=True)

print("Environment setup complete. All dependencies loaded.")

Environment setup complete. All dependencies loaded.


### 1. Data Ingestion & Structural Inspection

In [2]:
DATA_PATH = 'data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)

print(f"Raw Dataset Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns.")
df[['movie_name', 'Reviews', 'Ratings', 'genres', 'emotion']].head(3)

Raw Dataset Loaded: 46,173 rows x 45 columns.


,movie_name,Reviews,Ratings,genres,emotion
0,Waiting to Exhale,"It had some laughs, but overall the motivation of the characters was incomprehensible. Why should they be mad at men for cheating when they sleep with all sorts of married men themselves? Very hypocritical. Their lives are messed up because they messed them up with stupid choices. I had no empathy for any of these women.",3.0,"['Comedy', 'Drama', 'Romance']",anticipation
1,Waiting to Exhale,"WAITING TO EXHALE Waiting, and waiting, and waiting, and waiting... you get the point. ""Waiting To Exhale"", Forrest Whitaker's take on Terry McMillan's popular book, had a rather popular following upon it's release in 1995. It was packaged brilliantly, crossing over into the popular music scene with a blockbuster soundtrack featuring it's star Whitney Houston. However, as Leonard Maltin said it so beautifully, this film ultimately reminds one too much of the easy listening jazz that plays under nearly every scene.""Waiting To Exhale"" had the potential to be an interesting movie. It features a nice ensemble that manages to have good chemistry while also allowing certain performers to step into the limelight and really dominate certain scenes. Unfortunately, in the end, the movie is a repetitive drone.It tells the story of four African-American females (played by Angela Bassett, Whitney Houston, Loretta Divine, and Lela Rochon) as they struggle to find the men in life that can satisfy there needs. The only problem, in the world of this movie, men are nothing but complete ass-holes who wouldn't know the word ""feelings"" if they looked it up in the dictionary. How can this film possibly go anywhere when it's screenwriters has made men so incredibly unredeemable that nothing can change.For the first 45 minutes, the film is slightly enjoyable. However, as it continues on into it's 2 hour and plus running time... it begins to feel like deja-vu. The women keep putting themselves in identical situations to those they've experienced in the past... and as much as they talk about it in slow/sultry voice-overs, they don't seem to learn squat.It's like the soundtrack music. Slightly soothing, enjoyable, and easy to digest... but too slow and pointless to listen to for very long. ""Waiting To Exhale"" in the end is nothing more then a boringly pointless film that wastes the potential it had with the cast. Were the film given more of a focal point, and a more distinct narrative line, perhaps it could have been a good film. But everyone on board apparently missed the memo that... films are better when they have a plot and a purpose.... D ...",4.0,"['Comedy', 'Drama', 'Romance']",anticipation
2,Waiting to Exhale,"Angela Basset was good as expected, but Whitney has no Range as an actress. The screenplay also neglected to portray, on film, the greatness of this novel. Instead of promoting sisterhood, they emphasized the canine-qualities of men. Read the book; rent Soul Food instead!",4.0,"['Comedy', 'Drama', 'Romance']",anticipation


### 2. Sequential Execution of All 6 Preprocessing Stages

In [3]:
# ==============================================================================
# STAGE 1 (Member 1: Athapaththu A. M. P. P. - IT25102549)
# Text Cleaning, Noise Removal & Contraction Expansion
# ==============================================================================
CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'s": " is", "'re": " are", "'ve": " have", "'d": " would",
    "'ll": " will", "'m": " am", "it's": "it is"
}

def clean_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    for c, exp in CONTRACTIONS.items():
        text = text.replace(c, exp)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

df['cleaned_review'] = df['Reviews'].apply(clean_text)
print("✔ Stage 1 Complete: Text cleaned, noise stripped, contractions expanded.")

# ==============================================================================
# STAGE 2 (Member 2: Nishara W.A.S. - IT25102550)
# Domain-Specific Stopword Filtering & Lemmatization
# ==============================================================================
DOMAIN_STOPS = {
    'movie', 'movies', 'film', 'films', 'watch', 'watching',
    'one', 'like', 'really', 'see', 'saw', 'story', 'time',
    'character', 'characters', 'scene', 'scenes'
}
ALL_STOPS = set(ENGLISH_STOP_WORDS).union(DOMAIN_STOPS)

def filter_stopwords(text):
    tokens = text.split()
    return " ".join([t for t in tokens if t not in ALL_STOPS and len(t) > 2])

df['tokens_filtered'] = df['cleaned_review'].apply(filter_stopwords)
print("✔ Stage 2 Complete: Domain-specific stopwords filtered and vocabulary normalized.")

# ==============================================================================
# STAGE 3 (Member 3: Fernando B. K. H. - IT25102631)
# Categorical Multi-Label Encoding (Genres)
# ==============================================================================
def parse_genre_list(g_str):
    if pd.isna(g_str): return []
    try: return ast.literal_eval(str(g_str))
    except: return [x.strip(" '[]\"") for x in str(g_str).split(',') if x.strip()]

df['genre_list'] = df['genres'].apply(parse_genre_list)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_cols = [f"genre_{g.lower().replace(' ', '_').replace('-', '_')}" for g in mlb.classes_]
genre_df = pd.DataFrame(genre_matrix, columns=genre_cols)
df = pd.concat([df, genre_df], axis=1)
print(f"✔ Stage 3 Complete: {len(genre_cols)} binary genre indicator columns generated.")

# ==============================================================================
# STAGE 4 (Member 4: Abdullah H.F. - IT25102877)
# Numerical Cleaning, Outlier IQR Capping & Feature Scaling
# ==============================================================================
df['word_count'] = df['cleaned_review'].apply(lambda x: len(x.split()))
q1 = df['word_count'].quantile(0.25)
q3 = df['word_count'].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
lower_fence = max(0, q1 - 1.5 * iqr)

df['word_count_capped'] = df['word_count'].clip(lower=lower_fence, upper=upper_fence)

robust_scaler = RobustScaler()
df['word_count_robust'] = robust_scaler.fit_transform(df[['word_count_capped']])

minmax = MinMaxScaler()
df['rating_scaled'] = minmax.fit_transform(df[['Ratings']])
print(f"✔ Stage 4 Complete: Word count capped at {upper_fence:.1f} words. RobustScaler and MinMaxScaler applied.")

# ==============================================================================
# STAGE 5 (Member 5: Sameeha M.S.F. - IT25103066)
# Class Imbalance Quantification & Balanced Penalty Weights
# ==============================================================================
classes = np.array(np.unique(df['emotion']), dtype=str)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=df['emotion'].to_numpy())
class_weights = dict(zip(classes, weights.round(3)))
print("✔ Stage 5 Complete: Balanced class penalty weights computed for cost-sensitive training:")
for k, v in class_weights.items():
    print(f"   • {k}: {v}x loss penalty")

# ==============================================================================
# STAGE 6 (Member 6: Silva A.M.K.N. - IT25103132)
# Feature Extraction (TF-IDF) & Latent Semantic Analysis (TruncatedSVD)
# ==============================================================================
tfidf_pipe = TfidfVectorizer(max_features=2500, ngram_range=(1, 2), sublinear_tf=True, stop_words='english', min_df=3)
tfidf_pipe_matrix = tfidf_pipe.fit_transform(df['tokens_filtered'])

svd_pipe = TruncatedSVD(n_components=10, random_state=42)
svd_feats = svd_pipe.fit_transform(tfidf_pipe_matrix)
for i in range(10):
    df[f'lsa_component_{i+1}'] = svd_feats[:, i].round(4)
print("✔ Stage 6 Complete: Top 10 orthogonal LSA components extracted via TruncatedSVD.")

✔ Stage 1 Complete: Text cleaned, noise stripped, contractions expanded.
✔ Stage 2 Complete: Domain-specific stopwords filtered and vocabulary normalized.
✔ Stage 3 Complete: 20 binary genre indicator columns generated.
✔ Stage 4 Complete: Word count capped at 533.0 words. RobustScaler and MinMaxScaler applied.
✔ Stage 5 Complete: Balanced class penalty weights computed for cost-sensitive training:
   • anger: 1.586x loss penalty
   • anticipation: 0.787x loss penalty
   • disgust: 3.456x loss penalty
   • fear: 1.668x loss penalty
   • joy: 0.734x loss penalty
   • optimism: 1.199x loss penalty
   • sadness: 0.333x loss penalty
   • surprise: 101.257x loss penalty
✔ Stage 6 Complete: Top 10 orthogonal LSA components extracted via TruncatedSVD.


### 3. Pipeline Export for Phase 2 Model Training

In [4]:
# Select final feature columns for model training
OUTPUT_PATH = 'results/outputs/processed_movie_reviews.csv'

export_cols = [
    'Ratings', 'rating_scaled', 'word_count', 'word_count_capped', 'word_count_robust',
    'emotion', 'cleaned_review', 'tokens_filtered'
] + genre_cols + [f'lsa_component_{i+1}' for i in range(10)]

export_df = df[export_cols]
export_df.to_csv(OUTPUT_PATH, index=False)

print(f"✔ Pipeline successfully completed!")
print(f"Processed dataset saved to: {OUTPUT_PATH}")
print(f"Final Dimensions: {export_df.shape[0]:,} samples x {export_df.shape[1]} attributes.")
export_df.head(3)

✔ Pipeline successfully completed!
Processed dataset saved to: /Users/abdullahfawmy/Desktop/AI:ML Project/results/outputs/processed_movie_reviews.csv
Final Dimensions: 46,173 samples x 38 attributes.


,Ratings,rating_scaled,word_count,word_count_capped,word_count_robust,emotion,cleaned_review,tokens_filtered,genre_action,genre_adventure,genre_animation,genre_comedy,genre_crime,genre_documentary,genre_drama,genre_family,genre_fantasy,genre_foreign,genre_history,genre_horror,genre_music,genre_mystery,genre_romance,genre_science_fiction,genre_tv_movie,genre_thriller,genre_war,genre_western,lsa_component_1,lsa_component_2,lsa_component_3,lsa_component_4,lsa_component_5,lsa_component_6,lsa_component_7,lsa_component_8,lsa_component_9,lsa_component_10
0,3.0,0.222222,56,56,-0.765432,anticipation,it had some laughs but overall the motivation of the characters was incomprehensible why should they be mad at men for cheating when they sleep with all sorts of married men themselves very hypocritical their lives are messed up because they messed them up with stupid choices i had no empathy for any of these women,laughs overall motivation incomprehensible mad men cheating sleep sorts married men hypocritical lives messed messed stupid choices empathy women,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0.0628,-0.0298,-0.0138,-0.0014,-0.0222,0.0017,0.0715,-0.0011,-0.0157,-0.0095
1,4.0,0.333333,373,373,1.191358,anticipation,waiting to exhale waiting and waiting and waiting and waiting you get the point waiting to exhale forrest whitaker is take on terry mcmillan is popular book had a rather popular following upon it is release in it was packaged brilliantly crossing over into the popular music scene with a blockbuster soundtrack featuring it is star whitney houston however as leonard maltin said it so beautifully this film ultimately reminds one too much of the easy listening jazz that plays under nearly every scene waiting to exhale had the potential to be an interesting movie it features a nice ensemble that manages to have good chemistry while also allowing certain performers to step into the limelight and really dominate certain scenes unfortunately in the end the movie is a repetitive drone it tells the story of four african american females played by angela bassett whitney houston loretta divine and lela rochon as they struggle to find the men in life that can satisfy there needs the only problem in the world of this movie men are nothing but complete ass holes who would not know the word feelings if they looked it up in the dictionary how can this film possibly go anywhere when it is screenwriters has made men so incredibly unredeemable that nothing can change for the first minutes the film is slightly enjoyable however as it continues on into it is hour and plus running time it begins to feel like deja vu the women keep putting themselves in identical situations to those they have experienced in the past and as much as they talk about it in slow sultry voice overs they do not seem to learn squat it is like the soundtrack music slightly soothing enjoyable and easy to digest but too slow and pointless to listen to for very long waiting to exhale in the end is nothing more then a boringly pointless film that wastes the potential it had with the cast were the film given more of a focal point and a more distinct narrative line perhaps it could have been a good film but everyone on board apparently missed the memo that films are better when they have a plot and a purpose d,waiting exhale waiting waiting waiting waiting point waiting exhale forrest whitaker terry mcmillan popular book popular following release packaged brilliantly crossing popular music blockbuster soundtrack featuring star whitney houston leonard maltin said beautifully ultimately reminds easy listening jazz plays nearly waiting exhale potential interesting features nice ensemble manages good chemistry allowing certain performers step limelight dominate certain unfortunately end repetitive drone tells african american females played angela bassett whitney houston loretta divine lela rochon struggle men life satisfy needs problem world men complete ass holes know word feelings lo